# Use Delta Lake in Azure Databricks

In [ ]:
 %sql
 CREATE VOLUME IF NOT EXISTS product_data_volume

In [ ]:
 import requests

 # Download the CSV file
 url = "https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/products.csv"
 response = requests.get(url)
 response.raise_for_status()

 # Get the current catalog
 catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

 # Write directly to Unity Catalog volume
 volume_path = f"/Volumes/{catalog_name}/default/product_data_volume/products.csv"
 with open(volume_path, "wb") as f:
     f.write(response.content)

In [ ]:
df = spark.read.load(volume_path, format='csv', header=True)
display(df.limit(10))

In [ ]:
 # Create the table if it does not exist. Otherwise, replace the existing table.
 df.writeTo("Products").createOrReplace()

In [ ]:
 from delta.tables import DeltaTable

 # Reference the Delta table in Unity Catalog
 deltaTable = DeltaTable.forName(spark, "Products")

 # Perform the update
 deltaTable.update(
     condition = "ProductID == 771",
     set = { "ListPrice": "ListPrice * 0.9" })

 # View the updated data as a dataframe
 deltaTable.toDF().show(10)

In [ ]:
 new_df = spark.read.option("versionAsOf", 0).table("Products")
 new_df.show(10)

In [ ]:
deltaTable.history(10).show(10, False, True)

In [ ]:
 spark.sql("OPTIMIZE Products")
 spark.sql("VACUUM Products RETAIN 168 HOURS") # 7 days